In [ ]:
import pandas as pd

In [ ]:
raw_df = pd.read_csv('raw_data.csv')
df_data = raw_df.iloc[1:].copy()
df_data.columns = df_data.columns.str.strip()
df_long = df_data.melt(id_vars=['Capital IQ Information'],
                       var_name='Ticker',
                       value_name='Value')
df_long['Value'] = df_long['Value'].str.replace(',', '').str.strip()
df_long['Value'] = pd.to_numeric(df_long['Value'], errors='coerce')
df_median = df_long[df_long['Capital IQ Information'].str.contains('Revenue Median Consensus Estimate')].copy()
df_median['Year'] = df_median['Capital IQ Information'].str.extract(r'CY (\d{4})')
df_pivot = df_median.pivot(index=['Ticker', 'Year'],
                           columns='Capital IQ Information',
                           values='Value').reset_index()
df_pivot.columns.name = None
# Melt to long format
df_long = df_pivot.melt(
    id_vars=['Ticker', 'Year'],
    value_vars=[
        ' Revenue Median Consensus Estimate CY 2025 ',
        ' Revenue Median Consensus Estimate CY 2026 ',
        ' Revenue Median Consensus Estimate CY 2027 ',
        ' Revenue Median Consensus Estimate CY 2028 ',
        ' Revenue Median Consensus Estimate CY 2029 '
    ],
    var_name='Forecast_Type',
    value_name='RevenueForecast'
)

# Remove rows with missing forecasts
df_long = df_long.dropna(subset=['RevenueForecast'])

# Clean column names
df_long.columns = ['Ticker', 'Year', 'Forecast_Type', 'RevenueForecast']

# Convert RevenueForecast to float (remove any residual strings, commas, etc.)
df_long['RevenueForecast'] = pd.to_numeric(df_long['RevenueForecast'], errors='coerce')
df_long.drop(columns=['Forecast_Type'], inplace=True)
# Final check
print(df_long.head(10))
